In [1]:
import heapq
import numpy as np
from scipy.sparse import load_npz
from scipy.spatial import cKDTree

# 1. Nạp dữ liệu đồ thị từ A1
adj_matrix = load_npz("data/hcm_center_moto_adj.npz")
coords = np.load("data/hcm_center_moto_meta.npz")["coords"]
num_nodes = adj_matrix.shape[0]
tree = cKDTree(coords)

rows, cols = adj_matrix.nonzero()
base_weights = adj_matrix.data

# Ước lượng khoảng cách địa lý (mét) của từng cạnh để phân tách đường trục chính vs đường nhánh/hẻm
lat_u, lng_u = coords[rows, 0], coords[rows, 1]
lat_v, lng_v = coords[cols, 0], coords[cols, 1]
dy = (lat_v - lat_u) * 111320.0
dx = (lng_v - lng_u) * 111320.0 * np.cos(np.radians((lat_u + lat_v) / 2.0))
dist_m = np.sqrt(dx**2 + dy**2)
is_major_road = (dist_m / np.maximum(base_weights, 0.1)) > 5.5

# 2. Hàm hệ số kẹt xe theo khung giờ (đảm bảo tính chất FIFO)
def get_traffic_multiplier(hour_float, is_major):
    is_rush_hour = (7.0 <= hour_float <= 9.0) or (16.5 <= hour_float <= 19.0)
    is_moderate = (11.5 <= hour_float <= 13.5) or (19.0 < hour_float <= 21.0)
    if is_rush_hour:
        return 1.85 if is_major else 1.25
    elif is_moderate:
        return 1.30 if is_major else 1.10
    return 1.00

# 3. Time-Dependent Dijkstra
def time_dependent_dijkstra(num_nodes, rows, cols, base_weights, is_major_road, start_node, depart_hour):
    adj = [[] for _ in range(num_nodes)]
    for edge_idx, (u, v) in enumerate(zip(rows, cols)):
        adj[u].append((v, edge_idx))
        
    dist_sec = np.full(num_nodes, np.inf, dtype=np.float64)
    dist_sec[start_node] = 0.0
    pq = [(0.0, start_node)]
    
    while pq:
        elapsed_sec, u = heapq.heappop(pq)
        if elapsed_sec > dist_sec[u]:
            continue
        current_hour = (depart_hour + elapsed_sec / 3600.0) % 24.0
        for v, edge_idx in adj[u]:
            w_dynamic = base_weights[edge_idx] * get_traffic_multiplier(current_hour, is_major_road[edge_idx])
            new_elapsed = elapsed_sec + w_dynamic
            if new_elapsed < dist_sec[v]:
                dist_sec[v] = new_elapsed
                heapq.heappush(pq, (new_elapsed, v))
    return dist_sec / 60.0

# 4. So sánh 10:00 Sáng vs 17:30 Chiều
people_gps = {
    "An (Chợ Bến Thành)": (10.7725, 106.6980),
    "Bình (Hồ Con Rùa)": (10.7827, 106.6959),
    "Chi (Hàng Xanh)": (10.8013, 106.7114),
    "Phi (Vạn Kiếp)": (10.7995, 106.6922),
}
sources = [int(tree.query(gps)[1]) for gps in people_gps.values()]

results_td = {}
for label, hour in [("10:00 Sáng (Thấp điểm)", 10.0), ("17:30 Chiều (Cao điểm)", 17.5)]:
    T_hour = np.array([
        time_dependent_dijkstra(num_nodes, rows, cols, base_weights, is_major_road, s, hour)
        for s in sources
    ])
    F_sum = np.sum(T_hour, axis=0)
    F_max = np.max(T_hour, axis=0)
    best_sum_v, best_mm_v = int(np.argmin(F_sum)), int(np.argmin(F_max))
    results_td[label] = (best_sum_v, best_mm_v)
    
    print(f"=== {label} ===")
    print(f"  - Tối ưu Sum    : Đỉnh #{best_sum_v:<5} | Tổng = {F_sum[best_sum_v]:.1f}p | Từng người: {[round(x, 1) for x in T_hour[:, best_sum_v]]}")
    print(f"  - Tối ưu Minimax: Đỉnh #{best_mm_v:<5} | Max  = {F_max[best_mm_v]:.1f}p | Từng người: {[round(x, 1) for x in T_hour[:, best_mm_v]]}\n")

v_10h, v_17h30 = results_td["10:00 Sáng (Thấp điểm)"][1], results_td["17:30 Chiều (Cao điểm)"][1]
shift_m = np.linalg.norm((coords[v_17h30] - coords[v_10h]) * np.array([111320.0, 109350.0]))
print(f"-> Điểm hẹn Minimax dịch chuyển {shift_m:.1f} mét khi bước vào giờ cao điểm 17:30!")

=== 10:00 Sáng (Thấp điểm) ===
  - Tối ưu Sum    : Đỉnh #1828  | Tổng = 28.2p | Từng người: [np.float64(7.9), np.float64(0.1), np.float64(10.1), np.float64(10.1)]
  - Tối ưu Minimax: Đỉnh #1813  | Max  = 8.8p | Từng người: [np.float64(8.7), np.float64(3.5), np.float64(8.7), np.float64(8.8)]

=== 17:30 Chiều (Cao điểm) ===
  - Tối ưu Sum    : Đỉnh #1828  | Tổng = 40.3p | Từng người: [np.float64(10.9), np.float64(0.1), np.float64(15.2), np.float64(14.0)]
  - Tối ưu Minimax: Đỉnh #1671  | Max  = 13.0p | Từng người: [np.float64(11.5), np.float64(3.8), np.float64(13.0), np.float64(12.9)]

-> Điểm hẹn Minimax dịch chuyển 329.2 mét khi bước vào giờ cao điểm 17:30!
